# Stage 1 data: ground truth and C1 detections for training and testing C2/C3

Exports, for `boston_c23` (100 scenes, C2/C3 training) and `singapore_test` (50 scenes), per keyframe:
the timestamp, the global ego pose of the CAM_FRONT sample, every ground-truth object (class, instance,
global position, yaw, size, whether it is visible in CAM_FRONT), and the detections of the four Stage 0
C1 models (loaded from the Stage 0 kernel output, not retrained). C2 and C3 are then trained locally.

In [ ]:
import os, sys, glob, json, time, tarfile, hashlib, shutil, subprocess
from pathlib import Path
T0 = time.time()
def clock(m): print(f'[{time.time()-T0:7.1f}s] {m}', flush=True)

# --- labsd package (from ifty1011/labsd-src) ---
LABSD_DIR = '/kaggle/working/labsd_pkg'; Path(LABSD_DIR).mkdir(parents=True, exist_ok=True)
def locate_labsd():
    for init in glob.glob('/kaggle/input/**/labsd/__init__.py', recursive=True):
        return str(Path(init).parent.parent)
    for tar in glob.glob('/kaggle/input/**/labsd.tar', recursive=True):
        with tarfile.open(tar) as tf: tf.extractall(LABSD_DIR)
        for init in glob.glob(LABSD_DIR + '/**/labsd/__init__.py', recursive=True):
            return str(Path(init).parent.parent)
P = locate_labsd(); assert P, 'labsd not found'
if P not in sys.path: sys.path.insert(0, P)
import labsd
clock('labsd: ' + labsd.__file__)

# --- 350-scene subset (output of ifty1011/labsd-e1-subset350) ---
SPLITS = glob.glob('/kaggle/input/**/splits_350.json', recursive=True)
assert SPLITS, 'splits_350.json not found: attach kernel ifty1011/labsd-e1-subset350'
NUSC_ROOT = str(Path(SPLITS[0]).parent)
assert Path(NUSC_ROOT, 'v1.0-trainval', 'scene.json').exists()
splits = json.load(open(SPLITS[0]))
clock(f'subset root: {NUSC_ROOT}  parts: ' + str({k: len(v) for k, v in splits.items()}))

In [ ]:
clock('installing deps...')
pip = [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q']
subprocess.run(pip + ['--no-deps', 'nuscenes-devkit'], check=True)
subprocess.run(pip + ['pyquaternion', 'cachetools', 'descartes', 'shapely'], check=True)
subprocess.run(pip + ['--no-deps', 'ultralytics==8.4.105'], check=True)
subprocess.run(pip + ['ultralytics-thop', 'py-cpuinfo'], check=True)
import torch
assert torch.cuda.is_available(), 'no GPU: enable a GPU accelerator'
cap = torch.cuda.get_device_capability(0)
clock(f'GPU {torch.cuda.get_device_name(0)} sm_{cap[0]}{cap[1]}')
assert cap[0] >= 7, 'GPU below sm_70 (P100): select T4 so training does not fall back to CPU'
# The subset has no maps/ folder, and the devkit checks every map image at start-up.
# Maps are not used here: build a working root that links the subset's metadata and
# images, and points every map record at a 1x1 placeholder image (tested locally).
from PIL import Image
WORK_ROOT = Path('/kaggle/working/nusc_root')
(WORK_ROOT / 'v1.0-trainval').mkdir(parents=True, exist_ok=True)
(WORK_ROOT / 'maps').mkdir(exist_ok=True)
Image.new('L', (1, 1)).save(WORK_ROOT / 'maps' / 'placeholder.png')
if not (WORK_ROOT / 'samples').exists(): os.symlink(Path(NUSC_ROOT) / 'samples', WORK_ROOT / 'samples')
for f in Path(NUSC_ROOT, 'v1.0-trainval').glob('*.json'):
    dst = WORK_ROOT / 'v1.0-trainval' / f.name
    if dst.exists() or dst.is_symlink(): continue
    if f.name == 'map.json':
        recs = json.load(open(f))
        for r in recs: r['filename'] = 'maps/placeholder.png'
        dst.write_text(json.dumps(recs))
    else: os.symlink(f, dst)
from nuscenes.nuscenes import NuScenes
NUSC = NuScenes(version='v1.0-trainval', dataroot=str(WORK_ROOT), verbose=False)
clock(f'NuScenes loaded: {len(NUSC.scene)} scenes in metadata')

In [ ]:
# Stage 0 models (reuse, never retrain)
MAN = glob.glob('/kaggle/input/**/models/manifest.json', recursive=True)
assert MAN, 'Stage 0 models not found: attach kernel ifty1011/labsd-e1-stage0-c1'
MODEL_DIR = Path(MAN[0]).parent
stage0 = json.load(open(MAN[0]))
for name, info in stage0['models'].items():
    p = MODEL_DIR / info['file']; assert p.exists(), p
    h = hashlib.sha256(open(p, 'rb').read()).hexdigest()
    assert h == info['sha256'], f'checksum mismatch for {name}'
clock('Stage 0 models verified: ' + ', '.join(stage0['models']))

In [ ]:
from pyquaternion import Quaternion
from labsd.c1_yolo import nusc_category_to_yolo, project_3d_box_to_2d, dump_detections, YOLO_CLASS_NAMES
CLS = {c: i for i, c in enumerate(YOLO_CLASS_NAMES)}

def yaw_of(q): return float(Quaternion(q).yaw_pitch_roll[0])

def export_scenes(part):
    scenes = []
    for st in splits[part]:
        sc = NUSC.get('scene', st); samples = []
        tok = sc['first_sample_token']
        while tok:
            s = NUSC.get('sample', tok)
            cam = NUSC.get('sample_data', s['data']['CAM_FRONT'])
            cs = NUSC.get('calibrated_sensor', cam['calibrated_sensor_token'])
            ep = NUSC.get('ego_pose', cam['ego_pose_token'])
            gt = []
            for at in s['anns']:
                a = NUSC.get('sample_annotation', at)
                c = nusc_category_to_yolo(a['category_name'])
                if c is None: continue
                vis = project_3d_box_to_2d(tuple(a['translation']), tuple(a['size']), tuple(a['rotation']),
                                           cs['camera_intrinsic'], cs, ep, (cam['width'], cam['height'])) is not None
                gt.append([a['instance_token'], CLS[c], round(a['translation'][0], 3), round(a['translation'][1], 3),
                           round(yaw_of(a['rotation']), 4), round(a['size'][0], 2), round(a['size'][1], 2), int(vis),
                           a['num_lidar_pts'] + a['num_radar_pts']])
            samples.append({'token': tok, 't': cam['timestamp'],
                            'ego': [round(ep['translation'][0], 3), round(ep['translation'][1], 3), round(yaw_of(ep['rotation']), 5)],
                            'gt': gt})
            tok = s['next']
        scenes.append({'scene': st, 'name': sc['name'], 'samples': samples})
    return scenes

OUT = Path('/kaggle/working/stage1_data'); OUT.mkdir(exist_ok=True)
GT_FIELDS = ['instance', 'cls', 'x', 'y', 'yaw', 'width', 'length', 'visible_cam_front', 'n_points']
DATA = {}
for part in ['boston_c23', 'singapore_test']:
    t = time.time()
    DATA[part] = {'part': part, 'gt_fields': GT_FIELDS, 'classes': YOLO_CLASS_NAMES,
                  'ego_fields': ['x', 'y', 'yaw (global, CAM_FRONT sample ego pose)'],
                  'scenes': export_scenes(part), 'detections': {}}
    n = sum(len(s['samples']) for s in DATA[part]['scenes'])
    clock(f'{part}: {len(DATA[part]["scenes"])} scenes, {n} keyframes in {time.time()-t:.0f}s')

In [ ]:
# Detections of the four C1 models (ego frame of the CAM_FRONT sample: x forward, y left)
for part in ['boston_c23', 'singapore_test']:
    for name, info in stage0['models'].items():
        t = time.time()
        recs = dump_detections(str(MODEL_DIR / info['file']), splits[part], NUSC, score_thresh=0.25)
        by = {}
        for r in recs:
            by.setdefault(r['sample_token'], []).append([CLS[r['cls']], round(r['x'], 3), round(r['y'], 3), round(r['score'], 4)])
        DATA[part]['detections'][name] = by
        clock(f'{part} {name}: {len(recs)} detections in {time.time()-t:.0f}s')
    DATA[part]['det_fields'] = ['cls', 'x', 'y', 'score']
    json.dump(DATA[part], open(OUT / f'{part}.json', 'w'))

# Consistency with Stage 0: singapore_test detection counts must match exactly
check = {n: sum(len(v) for v in DATA['singapore_test']['detections'][n].values()) for n in stage0['models']}
expected = {n: info['n_detections_test'] for n, info in stage0['models'].items()}
summary = {'singapore_test_detection_counts': check, 'stage0_counts': expected, 'match': check == expected,
           'sizes_MB': {p.name: round(p.stat().st_size / 1e6, 2) for p in OUT.glob('*.json')}}
json.dump(summary, open(OUT / 'stage1_summary.json', 'w'), indent=1)
print(json.dumps(summary, indent=1))
clock('=== STAGE 1 DATA DONE ===')